In [ ]:
import numpy as np 
from vosk import KaldiRecognizer , Model
import wave 
import librosa 
import soundfile as sf 
from IPython.display import Audio , display
import webrtcvad 
from scipy.signal import medfilt
import json
from scipy.io.wavfile import write

In [ ]:
def frame_generator(y_int16, sr, frame_duration=30):
    n = int(sr * frame_duration / 1000)
    for offset in range(0, len(y_int16), n):
        frame = y_int16[offset:offset+n]

        if len(frame) < n:
            pad = np.zeros(n - len(frame), dtype=np.int16)
            frame = np.concatenate([frame, pad])
        yield frame

def vad_filter_safe(y, sr=16000, mode=2, frame_duration=30):
    y = np.clip(y, -1.0, 1.0)
    y_int16 = (y * 32767).astype(np.int16)
    vad = webrtcvad.Vad(mode)
    speech_frames = []

    for i, frame in enumerate(frame_generator(y_int16, sr, frame_duration)):
        if vad.is_speech(frame.tobytes(), sr):
            speech_frames.append(frame)
            
    if not speech_frames:
        return np.array([], dtype=np.float32)
    speech = np.concatenate(speech_frames)
    speech = speech.astype(np.float32) / 32767.0
    return speech

def silence_removal(audio_path ,new_audio_path):
    y_cleaned , sr = librosa.load(audio_path, mono=True)
    y_vad = vad_filter_safe(y_cleaned, sr=16000, mode=2)
    sf.write(new_audio_path, y_vad, sr, subtype='PCM_16')

    

In [ ]:
def denoise(audio_path ,new_audio_path):
    noisy_audio,sampling_rate = librosa.load(audio_path, sr= 16000, mono = True)
    s_full , phase = librosa.magphase(librosa.stft(noisy_audio))
    noise_power = np.mean(s_full[:, :int(sampling_rate*0.1) ], axis=1) 
    mask = s_full > noise_power[:, None]
    mask = mask.astype(float)
    mask = medfilt(mask, kernel_size=(1,5))
    s_clean= s_full * mask
    y_clean = librosa.istft(s_clean*phase)
    mask = mask.astype(float)
    mask = medfilt(mask, kernel_size=(1,5))
    s_clean= s_full * mask
    y_clean = librosa.istft(s_clean*phase)
    sf.write(new_audio_path, y_clean, sampling_rate)

In [ ]:
def audio_test(audio_path):
    noisy_audio,sampling_rate = librosa.load(audio_path, sr= 16000, mono = True)
    raw_noisy_audio = (noisy_audio*32767).astype(np.int16).tobytes()
    recognizer.AcceptWaveform(raw_noisy_audio)
    final_result = json.loads(recognizer.FinalResult())
    full_transcript = final_result.get("text","")
    return full_transcript
    

In [ ]:
model_path = 'vosk-model-en-us-0.22'
model = Model(model_path)
recognizer = KaldiRecognizer(model,16000)
recognizer.SetWords(True)

In [ ]:
print(audio_test('audio_files/Standard recording 4.wav'))

In [ ]:
display(Audio('audio_files/Standard recording 4.wav'))

In [ ]:
silence_removal('audio_files/Standard recording 4.wav','audio_files/Standardrecording4_prepreocessed.wav')

In [ ]:
print(audio_test('audio_files/Standardrecording4_prepreocessed.wav'))

In [ ]:
display(Audio('audio_files/Standardrecording4_prepreocessed.wav'))

In [ ]:
denoise('audio_files/Standardrecording4_prepreocessed.wav','audio_files/Standardrecording4_prepreocessed.wav')

In [ ]:
display(Audio('audio_files/Standardrecording4_prepreocessed.wav'))

In [ ]:
print(audio_test('audio_files/Standardrecording4_prepreocessed.wav'))
#denoise is f'ed up 

In [ ]:
print(audio_test('audio_files/Standardrecording4_prepreocessed.wav'))

In [ ]:
print(audio_test('audio_files/Standard recording 3.wav'))

In [ ]:
silence_removal('audio_files/Standard recording 3.wav','audio_files/Standardrecording3_prepreocessed.wav')
print(audio_test('audio_files/Standardrecording3_prepreocessed.wav'))

### model doesn't handel abbreviations and none english words well 

In [ ]:
print(audio_test('audio_files/micrecording_output.wav'))

In [ ]:
silence_removal('audio_files/micrecording_output.wav','audio_files/noise_reduction_output.wav')
print(audio_test('audio_files/noise_reduction_output.wav'))

In [ ]:
from glob import glob

In [ ]:
audio_files = glob('test_audio_files/*.wav')
audio_files

In [ ]:
for i in audio_files:
    print(audio_test(i))
    #display(Audio(i))

In [ ]:
for i in audio_files:
    j=1
    silence_removal(i,f'test_audio_processed/audio{j}_processed.wav')
    print(audio_test(i))
    #display(Audio(i))
    j+=1

In [ ]:
# import speech_recognition as sr

# r = sr.Recognizer()

# def  speech_recog(FILE_PATH):rail
#     with sr.AudioFile(FILE_PATH) as source:
#         audio_data = r.record(source)
#         text = r.recognize_google(audio_data)
#     return text



In [ ]:
# for i in audio_files:
#     print(speech_recog(i))
#     display(Audio(i))

In [ ]:
print(audio_test('test_audio_files/Standard recording20.wav'))

In [ ]:
silence_removal('test_audio_files/Standard recording20.wav', 'test_audio_processed/Standard recording20.wav')

In [ ]:
print(audio_test('test_audio_files/Standard recording20.wav'))

In [ ]:
test_data = glob('archive/Carl/Carl/*.wav')
for i in test_data:
    print(audio_test(i))
    display(Audio(i))

In [ ]:
test_data = glob('archive/Abuela Carl/Abuela Carl/*.wav')
for i in test_data:
    print(audio_test(i))
    display(Audio(i))

In [ ]:
test_data = glob('archive/Hermana_Molly/Hermana_Molly/*.wav')
for i in test_data:
    print(audio_test(i))
    display(Audio(i))

In [ ]:
display(Audio('test_audio_files/Standard recording20.wav'))

In [ ]:
denoise('test_audio_files/Standard recording20.wav','test_audio_processed/Standard recording20.wav')
silence_removal('test_audio_processed/Standard recording20.wav', 'test_audio_processed/Standard recording20.wav')
display(Audio('test_audio_processed/Standard recording20.wav'))
print(audio_test('test_audio_processed/Standard recording20.wav'))

In [ ]:
denoise('test_audio_files/Standard recording20.wav','test_audio_processed/Standard recording20.wav')
display(Audio('test_audio_processed/Standard recording20.wav'))
print(audio_test('test_audio_processed/Standard recording20.wav'))

In [ ]:
test_data = glob('archive/Hermana_Molly/Hermana_Molly/*.wav')
for i in test_data:
    denoise(i,'test_audio_processed/processed.wav')
    print(audio_test('test_audio_processed/processed.wav'))
    display(Audio('test_audio_processed/processed.wav'))

In [ ]:
test_data = glob('test_audio_files/*.wav')
for i in test_data:
    denoise(i,'test_audio_processed/Standard recording.wav')
    display(Audio(i))
    print(audio_test('test_audio_processed/Standard recording.wav'))